# **Preprocessing the CERT Dataset**:

### Imports & Constants:

In [1]:
from scripts.Preprocessing import *
from config import CERT_PATH, DEFAULT_OUTPUT_DIR, LDAP_PATH

In [2]:
WORK_HOURS = (9, 17)
LONG_URL_THRESHOLD = 90

### Building Layer A: (User, PC, Day) Level Dataset

In [3]:
layer_a_dataset, nunique_frames = build_layer_a(
    cert_path=CERT_PATH,
    work_hours=WORK_HOURS,
    return_nunique_frames=True
)

ldap_raw = load_ldap_profiles(LDAP_PATH, users=set(layer_a_dataset["user"]))
user_profiles = build_user_profiles(ldap_raw)
latest_snapshot = ldap_raw["_snapshot"].max()
print(f"User profiles built — {len(user_profiles)} users, {user_profiles['is_active'].sum()} active in latest snapshot ({latest_snapshot.strftime("%Y-%m")})")


Loading raw CERT logs (sample mode)...
Normalizing CERT logs...
  Normalizing logon.csv...
  Normalizing file.csv...
  Normalizing device.csv...
  email.csv: deferred (will normalize per-chunk)
  http.csv: deferred (will normalize per-chunk)
Extracting logon features...
Extracting file features...
Extracting device (USB) features...
Extracting email features (chunked)...
  Email chunk 1...
  Email chunk 2...
  Combining 2 email chunks...
Extracting HTTP features (chunked)...
  HTTP chunk 1...
  HTTP chunk 2...
  Combining 2 HTTP chunks...
  Building unique count domains...
Merging behavioral feature tables...
Adding PC behavioral features...
Layer A complete — 55,928 rows, 37 features.
Loaded 18 LDAP snapshots — 4000 unique users, 68923 total rows
User profiles built — 4000 users, 3639 active in latest snapshot (2011-05)


In [ ]:
layer_a_dataset.head()
layer_a_dataset.columns
ldap_raw
user_profiles
latest_snapshot

Index(['user', 'pc', 'day', 'logon_count', 'logoff_count', 'off_hours_logon',
       'file_open_count', 'file_write_count', 'file_copy_count',
       'file_delete_count', 'unique_files_accessed',
       'off_hours_files_accessed', 'usb_insert_count', 'usb_remove_count',
       'off_hours_usb_usage', 'emails_sent', 'external_emails_sent',
       'attachments_sent', 'off_hours_emails', 'unique_recipients',
       'http_total_requests', 'http_visit_count', 'http_download_count',
       'http_upload_count', 'http_jobsite_visits', 'http_cloud_storage_visits',
       'http_suspicious_site_visits', 'off_hours_http_requests',
       'http_long_url_count', 'unique_domains_visited', 'pc_prior_use_count',
       'pc_seen_before', 'pc_prior_use_ratio', 'pc_is_primary',
       'distinct_pcs_used_prior', 'n_pcs_used_today',
       'new_pc_after_stable_history'],
      dtype='object')

In [5]:
layer_a_path = save_dataset(
    dataset=layer_a_dataset,
    filename="ueba_dataset_4a.csv",
    output_dir=DEFAULT_OUTPUT_DIR
)

Dataset successfully saved to: /Users/melusi/Desktop/Projects/Machine Learning/Insider-Threat-Detection/processed_datasets/ueba_dataset_5/ueba_dataset_4a.csv


### Building Layer B: (User, Day) Model Training Dataset

In [6]:
layer_b_dataset = build_layer_b(
    layer_a_df=layer_a_dataset,
    rolling_window=5,
    nunique_frames=nunique_frames
)

Collapsing Layer A to (user, day) granularity...
  Aggregating per-PC behavioral features...
  Computing cross-channel flags...
  Recomputing true nunique counts from raw event frames...
Applying UEBA enhancements (z-scores, rolling deltas, risk flags)...
  Computing per-user causal z-score deviations...
  Computing causal rolling mean deltas...
  Adding cross-channel risk flags...
Layer B complete — 50,642 rows, 110 features.


In [7]:
layer_b_dataset.head()

,user,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,off_hours_files_accessed,...,non_primary_pc_http_requests_flag_rolling_delta,non_primary_pc_usb_flag_rolling_delta,non_primary_pc_file_copy_flag_rolling_delta,off_hours_activity_flag,usb_file_activity_flag,external_comm_activity_flag,jobsite_usb_activity_flag,suspicious_upload_flag,cloud_upload_flag,non_primary_pc_risk_flag
0,aab0162,2010-01-04,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0
1,aab0162,2010-01-05,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0
2,aab0162,2010-01-06,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
3,aab0162,2010-01-07,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,1,0,1,0,0,0,0
4,aab0162,2010-01-08,1,1,1,0,0,0,0,0,...,0.0,0.0,0.0,1,0,0,0,0,0,0


#### Enriching Layer B with LDAP User Profiles

In [8]:
layer_b_dataset = layer_b_dataset.merge(user_profiles, on="user", how="left")

layer_b_dataset["department"] = layer_b_dataset["department"].fillna("UNKNOWN")

_coverage = layer_b_dataset["employee_name"].notna().mean()
_unmatched = layer_b_dataset.loc[layer_b_dataset["employee_name"].isna(), "user"].nunique()
print(f"LDAP enrichment complete — {_coverage:.1%} of rows matched ({_unmatched} users unmatched)")

layer_b_dataset[["user", "employee_name", "role", "department", "functional_unit", "supervisor", "is_active"]].drop_duplicates("user").head()

LDAP enrichment complete — 100.0% of rows matched (0 users unmatched)


,user,employee_name,role,department,functional_unit,supervisor,is_active
0,aab0162,Amos Ahmed Burch,HardwareEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,Jeanette Macey Simpson,True
11,aab0398,Allegra April Bates,AdministrativeAssistant,None,9 - SalesAndMarketing_Government,Deborah Madaline Horton,True
22,aac0610,Ahmed Arsenio Cote,SoftwareQualityEngineer,3 - SoftwareManagement,3 - ResearchAndEngineering_Government_Domestic,Ann Tamekah Daniels,True
40,aac0668,August Amal Carter,MechanicalEngineer,4 - Engineering,3 - ResearchAndEngineering_Government_Domestic,Grace Jasmine Gamble,True
52,aac3270,Alexis Aretha Castro,StockroomClerk,3 - Assembly,5 - Manufacturing_Commercial,Quincy Guy Shannon,True


In [9]:
layer_b_path = save_dataset(
    dataset=layer_b_dataset,
    filename="ueba_dataset_4b.csv",
    output_dir=DEFAULT_OUTPUT_DIR
)

Dataset successfully saved to: /Users/melusi/Desktop/Projects/Machine Learning/Insider-Threat-Detection/processed_datasets/ueba_dataset_5/ueba_dataset_4b.csv


### Creating the Chronological Train/Test Split

In [10]:
# Splitting dataset B
train_df, test_df = chronological_split(df=layer_b_dataset, split_ratio=0.9)

In [11]:
# Saving training and testing sets
train_path = save_dataset(train_df, "ueba_dataset_4_train.csv", DEFAULT_OUTPUT_DIR)
test_path = save_dataset(test_df, "ueba_dataset_4_test_stream.csv", DEFAULT_OUTPUT_DIR)

Dataset successfully saved to: /Users/melusi/Desktop/Projects/Machine Learning/Insider-Threat-Detection/processed_datasets/ueba_dataset_5/ueba_dataset_4_train.csv
Dataset successfully saved to: /Users/melusi/Desktop/Projects/Machine Learning/Insider-Threat-Detection/processed_datasets/ueba_dataset_5/ueba_dataset_4_test_stream.csv


###  Utilizing Drill-Down Dataset Lookup

In [12]:
# Example usage
user = "abh0349"
day = "2010-01-27"

In [13]:
drill_down_table = get_drilldown(layer_a_dataset, user, day)

In [14]:
drill_down_table

,user,pc,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,...,off_hours_http_requests,http_long_url_count,unique_domains_visited,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_used_prior,n_pcs_used_today,new_pc_after_stable_history
